#  a

In [1]:
# ============================================================
# Phase 2: UA-ProtoNet Drop-One & Prioritized Ablation Pipeline
# Task 2.1 - FINAL / HARDENED VERSION (Kaggle GPU ready)
# ============================================================

import os
import random
import json
import glob
import contextlib
from collections import defaultdict
from dataclasses import dataclass
from typing import Dict, List, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models

# ============================================================
# 1. Dataset Paths & Configuration
# ============================================================

PLANTVILLAGE_ROOT = "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color"
PLANTDOC_ROOT = "/kaggle/input/datasets/tanvirakon/plantdoc-test-train-merged/combined"
TASK2_1_JSON_DIR = "/kaggle/input/datasets/tanvirakon/task2-1"

# Production continuation for the final ablation variant (reg_suite).
QUICK_DRY_RUN = False

# Schedule the final remaining variant.
VARIANTS_TO_RUN = ["reg_suite"]

# Valid completed metrics recovered from sessions 1 and 2.
COMPLETED_RESULTS = {
    "full": {"pv_acc": 93.47, "pd_seen": 53.69, "pd_unseen": 44.49},
    "no_dual_attn": {"pv_acc": 94.96, "pd_seen": 54.59, "pd_unseen": 44.88},
    "no_multiscale": {"pv_acc": 93.26, "pd_seen": 52.24, "pd_unseen": 44.11},
    "no_learnable_temp": {"pv_acc": 83.42, "pd_seen": 48.11, "pd_unseen": 36.83},
}

@dataclass
class AblationConfig:
    pv_root: str = PLANTVILLAGE_ROOT
    pd_root: str = PLANTDOC_ROOT
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    image_size: int = 224
    lr: float = 1e-3
    weight_decay: float = 1e-2
    epochs: int = 1 if QUICK_DRY_RUN else 40
    patience: int = 2 if QUICK_DRY_RUN else 8
    warmup_epochs: int = 0 if QUICK_DRY_RUN else 5

    train_episodes: int = 2 if QUICK_DRY_RUN else 600
    val_episodes: int = 2 if QUICK_DRY_RUN else 300
    test_episodes: int = 2 if QUICK_DRY_RUN else 600
    eval_pd_episodes: int = 2 if QUICK_DRY_RUN else 300

    n_way: int = 5
    k_shot: int = 5
    q_query: int = 15
    mc_dropout_T: int = 2 if QUICK_DRY_RUN else 10

    split_seed: int = 42
    results_dir: str = "/kaggle/working/ablation_results" if os.path.exists("/kaggle") else "./ablation_results"

CFG = AblationConfig()
os.makedirs(CFG.results_dir, exist_ok=True)

def set_seed(seed: int = 42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def autocast_context(device: str):
    if str(device).startswith("cuda"):
        try:
            return torch.amp.autocast(device_type="cuda")
        except AttributeError:
            return torch.cuda.amp.autocast()
    return contextlib.nullcontext()

def make_grad_scaler(use_cuda: bool):
    if not use_cuda:
        return None
    try:
        return torch.amp.GradScaler(device="cuda")
    except AttributeError:
        return torch.cuda.amp.GradScaler()

train_tfms = transforms.Compose([
    transforms.Resize((CFG.image_size, CFG.image_size)),
    transforms.RandomResizedCrop(CFG.image_size, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_tfms = transforms.Compose([
    transforms.Resize((CFG.image_size, CFG.image_size)),
    transforms.CenterCrop(CFG.image_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ============================================================
# 2. Modular Ablated Architecture
# ============================================================

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        reduced = max(1, in_planes // reduction)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, reduced, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(reduced, in_planes, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        mask = self.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))
        return x * mask


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        mask = self.sigmoid(self.conv1(torch.cat([avg_out, max_out], dim=1)))
        return x * mask


class DualAttention(nn.Module):
    def __init__(self, in_planes, reduction=16, kernel_size=7):
        super().__init__()
        self.ca = ChannelAttention(in_planes, reduction)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        return self.sa(self.ca(x))


class AblatedEfficientNetB0FSL(nn.Module):
    def __init__(self, ablation_variant: str = "full", pretrained: bool = True):
        super().__init__()
        self.ablation_variant = ablation_variant
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        effnet = models.efficientnet_b0(weights=weights)

        self.features_early = effnet.features[:7]
        self.features_late = effnet.features[7:]
        self.pooling = nn.AdaptiveAvgPool2d(1)

        if self.ablation_variant != "no_dual_attn":
            self.attention_mid = DualAttention(192, reduction=8)
            self.attention_final = DualAttention(1280, reduction=16)
        else:
            self.attention_mid = nn.Identity()
            self.attention_final = nn.Identity()

        if self.ablation_variant != "no_multiscale":
            self.mid_proj = nn.Linear(192, 128)
            self.late_proj = nn.Linear(1280, 128)
            self.proj = nn.Linear(256, 256)
        else:
            self.late_proj = nn.Linear(1280, 256)
            self.proj = nn.Linear(256, 256)

        p_drop = 0.0 if self.ablation_variant == "no_mc_dropout" else 0.5
        self.mcd_dropout = nn.Dropout(p=p_drop)

        if self.ablation_variant == "no_learnable_temp":
            self.temperature = 1.0
        else:
            self.temperature = nn.Parameter(torch.tensor(1.0))

    def forward(self, x):
        x = self.features_early(x)
        x_mid = self.attention_mid(x)
        x = self.features_late(x_mid)
        x_late = self.attention_final(x)

        if self.ablation_variant != "no_multiscale":
            mid_feat = torch.flatten(self.pooling(x_mid), 1)
            late_feat = torch.flatten(self.pooling(x_late), 1)
            combined = torch.cat([self.mid_proj(mid_feat), self.late_proj(late_feat)], dim=1)
        else:
            combined = self.late_proj(torch.flatten(self.pooling(x_late), 1))

        return F.normalize(self.proj(self.mcd_dropout(combined)), dim=1)

    def enable_mcdropout(self):
        if self.ablation_variant != "no_mc_dropout":
            self.mcd_dropout.train()

# ============================================================
# 3. Regularization Losses & Augmentations
# ============================================================

class ContrastiveLoss(nn.Module):
    def __init__(self, margin: float = 1.0):
        super().__init__()
        self.margin = margin

    def forward(self, embeddings, labels):
        dist_matrix = torch.cdist(embeddings, embeddings, p=2)
        label_eq = (labels.unsqueeze(0) == labels.unsqueeze(1)).float()
        mask_upper = torch.triu(torch.ones_like(dist_matrix, dtype=torch.bool), diagonal=1)
        pos_pairs = label_eq * dist_matrix
        neg_pairs = (1 - label_eq) * F.relu(self.margin - dist_matrix)
        loss = (pos_pairs[mask_upper].sum() + neg_pairs[mask_upper].sum()) / mask_upper.sum().clamp(min=1)
        return loss


def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    perm = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[perm]
    return mixed_x, y, y[perm], lam


def cutmix_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    B, C, H, W = x.shape
    perm = torch.randperm(B, device=x.device)
    cut_rat = np.sqrt(1.0 - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = int(np.clip(cx - cut_w // 2, 0, W))
    x2 = int(np.clip(cx + cut_w // 2, 0, W))
    y1 = int(np.clip(cy - cut_h // 2, 0, H))
    y2 = int(np.clip(cy + cut_h // 2, 0, H))
    mixed_x = x.clone()
    mixed_x[:, :, y1:y2, x1:x2] = x[perm, :, y1:y2, x1:x2]
    lam = 1 - ((x2 - x1) * (y2 - y1) / (W * H))
    return mixed_x, y, y[perm], lam


def mixup_cross_entropy(logits, y_a, y_b, lam, label_smoothing=0.1):
    return (lam * F.cross_entropy(logits, y_a, label_smoothing=label_smoothing) +
            (1 - lam) * F.cross_entropy(logits, y_b, label_smoothing=label_smoothing))


def orthogonal_reg(prototypes, weight=0.01):
    proto_norm = F.normalize(prototypes, dim=1)
    similarity = torch.mm(proto_norm, proto_norm.t())
    identity = torch.eye(similarity.size(0), device=similarity.device)
    return ((similarity - identity) ** 2).sum() * weight


def compute_prototypes(embeddings, labels, n_way):
    protos = []
    for c in range(n_way):
        cls_emb = embeddings[labels == c]
        if cls_emb.size(0) == 0:
            protos.append(torch.zeros(embeddings.size(1), device=embeddings.device))
        else:
            protos.append(cls_emb.mean(dim=0))
    return torch.stack(protos)


def euclidean_logits(query_embeddings, prototypes, temperature=1.0):
    if isinstance(temperature, torch.Tensor):
        # Keep temperature in the autograd graph so the learnable-tau ablation is real.
        temp = temperature.clamp_min(1e-6)
    else:
        temp = max(float(temperature), 1e-6)
    return -torch.cdist(query_embeddings, prototypes, p=2) / temp

# ============================================================
# 4. Dataset Helpers & Fixed Episode Loading
# ============================================================

class ImageFolderWithIndex(datasets.ImageFolder):
    def __getitem__(self, index):
        img, label = super().__getitem__(index)
        return img, label, index

class FewShotEpisodeDataset(Dataset):
    def __init__(self, base_dataset, valid_classes, class_to_indices, n_way, k_shot, q_query, episodes):
        self.base_dataset = base_dataset
        self.valid_classes = list(valid_classes)
        self.class_to_indices = class_to_indices
        self.n_way = n_way
        self.k_shot = k_shot
        self.q_query = q_query
        self.episodes = episodes

        if len(self.valid_classes) < n_way:
            raise ValueError(f"Not enough valid classes ({len(self.valid_classes)}) for n_way={n_way}.")
        need = k_shot + q_query
        self.valid_classes = [c for c in self.valid_classes if len(self.class_to_indices.get(c, [])) >= need]
        if len(self.valid_classes) < n_way:
            raise ValueError(f"After filtering, only {len(self.valid_classes)} classes have >= {need} images.")

    def __len__(self):
        return self.episodes

    def __getitem__(self, _):
        sampled_classes = random.sample(self.valid_classes, self.n_way)
        support_x, support_y, query_x, query_y = [], [], [], []
        for mapped_label, cls in enumerate(sampled_classes):
            img_indices = random.sample(self.class_to_indices[cls], self.k_shot + self.q_query)
            for i in range(self.k_shot):
                img, _, _ = self.base_dataset[img_indices[i]]
                support_x.append(img); support_y.append(mapped_label)
            for i in range(self.k_shot, self.k_shot + self.q_query):
                img, _, _ = self.base_dataset[img_indices[i]]
                query_x.append(img); query_y.append(mapped_label)
        return (torch.stack(support_x), torch.tensor(support_y, dtype=torch.long),
                torch.stack(query_x), torch.tensor(query_y, dtype=torch.long))

class FixedEpisodeDataset(Dataset):
    def __init__(self, base_dataset, episodes: List[dict]):
        self.base_dataset = base_dataset
        self.dataset_len = len(base_dataset)
        self.episodes = self._validate(episodes)

    def _validate(self, episodes):
        valid = []
        dropped = 0
        for ep in episodes:
            idxs = list(ep["support_indices"]) + list(ep["query_indices"])
            if all(0 <= int(i) < self.dataset_len for i in idxs):
                valid.append(ep)
            else:
                dropped += 1
        if dropped:
            print(f"   ⚠️ Dropped {dropped}/{len(episodes)} episodes with out-of-range indices "
                  f"(dataset size={self.dataset_len}).", flush=True)
        if not valid:
            raise IndexError("All episodes reference indices outside this dataset. "
                             "JSON built on a different dataset/ordering than loaded here.")
        return valid

    def __len__(self):
        return len(self.episodes)

    def __getitem__(self, idx):
        ep = self.episodes[idx]
        support_x = torch.stack([self.base_dataset[int(i)][0] for i in ep["support_indices"]])
        query_x = torch.stack([self.base_dataset[int(i)][0] for i in ep["query_indices"]])
        support_y = torch.tensor(ep["support_labels"], dtype=torch.long)
        query_y = torch.tensor(ep["query_labels"], dtype=torch.long)
        return support_x, support_y, query_x, query_y


class FixedCrossDomainDataset(Dataset):
    def __init__(self, pv_dataset, pd_dataset, episodes: List[dict]):
        self.pv_dataset = pv_dataset
        self.pd_dataset = pd_dataset
        self.pv_len = len(pv_dataset)
        self.pd_len = len(pd_dataset)
        self.episodes = self._validate(episodes)

    def _validate(self, episodes):
        valid = []
        dropped = 0
        for ep in episodes:
            s_ok = all(0 <= int(i) < self.pv_len for i in ep["support_indices"])
            q_ok = all(0 <= int(i) < self.pd_len for i in ep["query_indices"])
            if s_ok and q_ok:
                valid.append(ep)
            else:
                dropped += 1
        if dropped:
            print(f"   ⚠️ Dropped {dropped}/{len(episodes)} episodes with out-of-range indices "
                  f"(PV len={self.pv_len}, PD len={self.pd_len}).", flush=True)
        if not valid:
            raise IndexError("All episodes reference indices outside the loaded datasets. "
                             "JSON built on different dataset paths or orderings than loaded here.")
        return valid

    def __len__(self):
        return len(self.episodes)

    def __getitem__(self, idx):
        ep = self.episodes[idx]
        support_x = torch.stack([self.pv_dataset[int(i)][0] for i in ep["support_indices"]])
        query_x = torch.stack([self.pd_dataset[int(i)][0] for i in ep["query_indices"]])
        support_y = torch.tensor(ep["support_labels"], dtype=torch.long)
        query_y = torch.tensor(ep["query_labels"], dtype=torch.long)
        return support_x, support_y, query_x, query_y


def find_pv_json(keyword: str) -> Optional[str]:
    hits = sorted(glob.glob(os.path.join(TASK2_1_JSON_DIR, f"*{keyword}*.json")))
    return hits[0] if hits else None

def find_pd_json(keyword: str) -> Optional[str]:
    hits = sorted(glob.glob(os.path.join(TASK2_1_JSON_DIR, f"*{keyword}*.json")))
    if keyword == "SEEN":
        # CRITICAL: "UNSEEN" contains "SEEN" - must exclude it
        hits = [h for h in hits if "UNSEEN" not in os.path.basename(h)]
    return hits[0] if hits else None

def load_or_create_episodes(split_name, valid_classes, class_to_indices, episodes, rng_seed, cfg):
    found = find_pv_json(f"pv_{split_name}")
    if found:
        with open(found, "r") as f:
            loaded = json.load(f)
        selected = loaded[:episodes]
        print(f"✓ Loaded {len(selected)}/{len(loaded)} fixed {split_name} episodes from: "
              f"{os.path.basename(found)}", flush=True)
        return selected

    filename = os.path.join(
        cfg.results_dir,
        f"pv_{split_name}_fixed_episodes_split{cfg.split_seed}_n{cfg.n_way}_k{cfg.k_shot}_q{cfg.q_query}_ep{episodes}.json"
    )
    if os.path.exists(filename):
        with open(filename, "r") as f:
            return json.load(f)[:episodes]

    need = cfg.k_shot + cfg.q_query
    usable = [c for c in valid_classes if len(class_to_indices.get(c, [])) >= need]
    if len(usable) < cfg.n_way:
        raise ValueError(f"[{split_name}] Only {len(usable)} classes have >= {need} images; need >= {cfg.n_way}.")

    rng = random.Random(rng_seed)
    episode_list = []
    for _ in range(episodes):
        sampled_classes = rng.sample(usable, cfg.n_way)
        s_idx, s_lab, q_idx, q_lab = [], [], [], []
        for mapped_label, cls in enumerate(sampled_classes):
            selected = rng.sample(class_to_indices[cls], need)
            s_idx.extend(selected[:cfg.k_shot]); s_lab.extend([mapped_label] * cfg.k_shot)
            q_idx.extend(selected[cfg.k_shot:]); q_lab.extend([mapped_label] * cfg.q_query)
        episode_list.append({"support_indices": s_idx, "support_labels": s_lab,
                             "query_indices": q_idx, "query_labels": q_lab})
    with open(filename, "w") as f:
        json.dump(episode_list, f)
    return episode_list

# ============================================================
# 5. Training Loop
# ============================================================

def train_ablation_fsl(model, train_loader, val_loader, cfg, variant):
    use_cuda = str(cfg.device).startswith("cuda")
    contrastive_fn = ContrastiveLoss(margin=1.0)

    head_params = list(model.proj.parameters()) + list(model.late_proj.parameters())
    if hasattr(model, 'mid_proj'):
        head_params += list(model.mid_proj.parameters())
    if not isinstance(model.attention_mid, nn.Identity):
        head_params += list(model.attention_mid.parameters()) + list(model.attention_final.parameters())
    if isinstance(model.temperature, nn.Parameter):
        head_params.append(model.temperature)

    backbone_params = list(model.features_early.parameters()) + list(model.features_late.parameters())

    optimizer = torch.optim.AdamW([
        {"params": head_params, "lr": cfg.lr},
        {"params": backbone_params, "lr": cfg.lr * 0.01}
    ], weight_decay=cfg.weight_decay)

    t_max = max(1, cfg.epochs - cfg.warmup_epochs)
    warmup_ep = min(cfg.warmup_epochs, cfg.epochs - 1) if cfg.epochs > 1 else 0

    if warmup_ep > 0:
        scheduler = torch.optim.lr_scheduler.SequentialLR(
            optimizer,
            schedulers=[
                torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, total_iters=warmup_ep),
                torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=t_max, eta_min=1e-6)
            ],
            milestones=[warmup_ep]
        )
    else:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, cfg.epochs), eta_min=1e-6)

    scaler = make_grad_scaler(use_cuda)

    cutmix_prob = 0.0 if variant in ["no_cutmix_mixup", "reg_suite"] else 0.5
    mixup_alpha = 0.0 if variant in ["no_cutmix_mixup", "reg_suite"] else 0.2
    label_smoothing = 0.0 if variant in ["no_label_smoothing", "reg_suite"] else 0.1
    ortho_weight = 0.0 if variant in ["no_ortho_reg", "reg_suite"] else 0.01
    contrastive_lambda = 0.0 if variant in ["no_contrastive_loss", "reg_suite"] else 0.1

    best_val_acc = -1.0
    epochs_no_improve = 0
    dry_suffix = "_dry" if QUICK_DRY_RUN else ""
    save_filename = os.path.join(
        cfg.results_dir,
        f"ablation_{variant}{dry_suffix}_seed_{cfg.split_seed}.pth"
    )

    for epoch in range(cfg.epochs):
        model.train()
        for support_x, support_y, query_x, query_y in train_loader:
            support_x, support_y = support_x.squeeze(0).to(cfg.device), support_y.squeeze(0).to(cfg.device)
            query_x, query_y = query_x.squeeze(0).to(cfg.device), query_y.squeeze(0).to(cfg.device)
            optimizer.zero_grad()

            if cutmix_prob > 0 and random.random() < cutmix_prob:
                mixed_qx, qy_a, qy_b, lam = cutmix_data(query_x, query_y, alpha=mixup_alpha)
            elif mixup_alpha > 0:
                mixed_qx, qy_a, qy_b, lam = mixup_data(query_x, query_y, alpha=mixup_alpha)
            else:
                mixed_qx, qy_a, qy_b, lam = query_x, query_y, query_y, 1.0

            with autocast_context(cfg.device):
                all_emb = model(torch.cat([support_x, mixed_qx], dim=0))
                support_emb, query_emb = all_emb[:support_x.size(0)], all_emb[support_x.size(0):]
                prototypes = compute_prototypes(support_emb, support_y, cfg.n_way)
                logits = euclidean_logits(query_emb, prototypes, model.temperature)

                ce_loss = mixup_cross_entropy(logits, qy_a, qy_b, lam, label_smoothing=label_smoothing)
                con_loss = contrastive_fn(support_emb, support_y) if contrastive_lambda > 0 else torch.tensor(0.0, device=cfg.device)
                ortho_loss = orthogonal_reg(prototypes, weight=ortho_weight) if ortho_weight > 0 else torch.tensor(0.0, device=cfg.device)
                loss = ce_loss + contrastive_lambda * con_loss + ortho_loss

            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()

        model.eval()
        val_acc = 0.0
        with torch.no_grad():
            for support_x, support_y, query_x, query_y in val_loader:
                support_x, support_y = support_x.squeeze(0).to(cfg.device), support_y.squeeze(0).to(cfg.device)
                query_x, query_y = query_x.squeeze(0).to(cfg.device), query_y.squeeze(0).to(cfg.device)
                with autocast_context(cfg.device):
                    all_emb = model(torch.cat([support_x, query_x], dim=0))
                    prototypes = compute_prototypes(all_emb[:support_x.size(0)], support_y, cfg.n_way)
                    logits = euclidean_logits(all_emb[support_x.size(0):], prototypes, model.temperature)
                val_acc += (logits.argmax(1) == query_y).float().mean().item()

        val_acc /= max(1, len(val_loader))
        scheduler.step()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_no_improve = 0
            torch.save(model.state_dict(), save_filename)
            print(f"   Epoch {epoch+1}/{cfg.epochs} - Val Acc: {val_acc*100:.2f}% (⭐ New Best Saved)", flush=True)
        else:
            epochs_no_improve += 1
            print(f"   Epoch {epoch+1}/{cfg.epochs} - Val Acc: {val_acc*100:.2f}%", flush=True)
            if epochs_no_improve >= cfg.patience:
                print(f"   Early stopping triggered at epoch {epoch+1}.", flush=True)
                break

    if os.path.exists(save_filename):
        try:
            model.load_state_dict(torch.load(save_filename, map_location=cfg.device, weights_only=True))
        except TypeError:
            model.load_state_dict(torch.load(save_filename, map_location=cfg.device))
    return model

# ============================================================
# 6. Evaluation & Table Generator
# ============================================================

def evaluate_ablation_model(model, test_loader, cfg):
    model.eval()
    det_accs = []
    with torch.no_grad():
        for support_x, support_y, query_x, query_y in test_loader:
            support_x = support_x.squeeze(0).to(cfg.device)
            support_y = support_y.squeeze(0).to(cfg.device)
            query_x = query_x.squeeze(0).to(cfg.device)
            query_y = query_y.squeeze(0).to(cfg.device)
            with autocast_context(cfg.device):
                all_emb = model(torch.cat([support_x, query_x], dim=0))
                prototypes = compute_prototypes(all_emb[:support_x.size(0)], support_y, cfg.n_way)
                logits = euclidean_logits(all_emb[support_x.size(0):], prototypes, model.temperature)
                probs = torch.softmax(logits.float(), dim=-1)
            det_accs.append((probs.argmax(dim=1) == query_y).float().mean().item())
    pv_acc = float(np.mean(det_accs)) * 100.0 if det_accs else float("nan")
    print(f"     [Evaluation] PV Acc: {pv_acc:.2f}%", flush=True)
    return {"pv_acc": pv_acc}


@torch.no_grad()
def evaluate_plantdoc_episodes(model, pd_loader, cfg, split_name):
    model.eval()
    accs = []
    total = len(pd_loader)
    for episode_idx, (support_x, support_y, query_x, query_y) in enumerate(pd_loader, start=1):
        support_x, support_y = support_x.squeeze(0).to(cfg.device), support_y.squeeze(0).to(cfg.device)
        query_x, query_y = query_x.squeeze(0).to(cfg.device), query_y.squeeze(0).to(cfg.device)
        with autocast_context(cfg.device):
            all_emb = model(torch.cat([support_x, query_x], dim=0))
            prototypes = compute_prototypes(all_emb[:support_x.size(0)], support_y, cfg.n_way)
            logits = euclidean_logits(all_emb[support_x.size(0):], prototypes, model.temperature)
        accs.append((logits.argmax(dim=1) == query_y).float().mean().item())
        if episode_idx % 25 == 0 or episode_idx == total:
            running_acc = float(np.mean(accs)) * 100.0
            print(f"     [{split_name}] {episode_idx}/{total} episodes | "
                  f"running acc: {running_acc:.2f}%", flush=True)
    return float(np.mean(accs)) * 100.0 if accs else float("nan")

def generate_streamlined_ablation_table(results):
    print("\n" + "=" * 95)
    print("📊 TASK 2.1 STREAMLINED 5-ROW ABLATION SUMMARY TABLE")
    print("=" * 95)
    print(f"{'Configuration / Variant':<32} | {'PV Test Acc':<12} | {'PDoc SEEN':<12} | {'PDoc UNSEEN':<12} | {'Δ vs Full (PV)':<14}")
    print("-" * 95)

    display_rows = [
        ("Full UA-ProtoNet (Baseline)", "full"),
        ("- Dual Attention (P0 Core)", "no_dual_attn"),
        ("- Multi-Scale Fusion (P0 Core)", "no_multiscale"),
        ("- Learnable tau (P0 Metric)", "no_learnable_temp"),
        ("- Regularization Suite (P1/P2)", "reg_suite"),
    ]

    if "full" not in results or "pv_acc" not in results["full"]:
        raise KeyError("'full' baseline results missing! Cannot compute ablation deltas.")
    full_pv = results["full"]["pv_acc"]

    for label, key in display_rows:
        m = results.get(key, {})
        pv_acc = m.get("pv_acc", float("nan"))
        pd_seen = m.get("pd_seen", float("nan"))
        pd_unseen = m.get("pd_unseen", float("nan"))
        delta = (pv_acc - full_pv) if not np.isnan(pv_acc) else float("nan")
        delta_str = f"{delta:+.2f}%" if not np.isnan(delta) else "N/A"
        pv_str = f"{pv_acc:.2f}%" if not np.isnan(pv_acc) else "N/A"
        seen_str = f"{pd_seen:.2f}%" if not np.isnan(pd_seen) else "N/A"
        unseen_str = f"{pd_unseen:.2f}%" if not np.isnan(pd_unseen) else "N/A"
        print(f"{label:<32} | {pv_str:<12} | {seen_str:<12} | {unseen_str:<12} | {delta_str:<14}")
    print("=" * 95)

# ============================================================
# 7. Main Execution
# ============================================================

def run_suite():
    if QUICK_DRY_RUN:
        print("\n" + "!" * 80)
        print("  ⚠️ QUICK_DRY_RUN = TRUE  (test mode, results NOT valid for thesis)")
        print("!" * 80 + "\n", flush=True)
    else:
        print("\n" + "=" * 80)
        print("  🚀 FULL THESIS PRODUCTION RUN (QUICK_DRY_RUN = False)")
        print("=" * 80 + "\n", flush=True)

    set_seed(CFG.split_seed)

    if not os.path.exists(CFG.pv_root):
        raise FileNotFoundError(f"❌ PlantVillage root not found at '{CFG.pv_root}'!")
    if not os.path.exists(CFG.pd_root):
        raise FileNotFoundError(f"❌ PlantDoc root not found at '{CFG.pd_root}'!")

    pd_seen_json_path = find_pd_json("SEEN")
    pd_unseen_json_path = find_pd_json("UNSEEN")
    if not pd_seen_json_path or not pd_unseen_json_path:
        raise FileNotFoundError("❌ PlantDoc SEEN/UNSEEN episode JSON not found.")
    print(f"✓ SEEN json:   {os.path.basename(pd_seen_json_path)}", flush=True)
    print(f"✓ UNSEEN json: {os.path.basename(pd_unseen_json_path)}", flush=True)

    full_ds = ImageFolderWithIndex(CFG.pv_root, transform=train_tfms)
    val_ds = ImageFolderWithIndex(CFG.pv_root, transform=val_tfms)

    class_to_indices = defaultdict(list)
    for idx, target in enumerate(full_ds.targets):
        class_to_indices[target].append(idx)

    n_classes = len(full_ds.classes)
    print(f"✓ PlantVillage classes detected: {n_classes}", flush=True)
    if n_classes < 38:
        raise ValueError(f"Expected >= 38 PlantVillage classes, found {n_classes}. Check path. Classes: {full_ds.classes}")

    all_classes = list(range(n_classes))
    random.Random(CFG.split_seed).shuffle(all_classes)
    train_classes = all_classes[:23]
    val_classes = all_classes[23:31]
    test_classes = all_classes[31:38]

    val_episodes = load_or_create_episodes("val", val_classes, class_to_indices, CFG.val_episodes, CFG.split_seed + 100001, CFG)
    test_episodes = load_or_create_episodes("test", test_classes, class_to_indices, CFG.test_episodes, CFG.split_seed + 200002, CFG)

    train_loader = DataLoader(FewShotEpisodeDataset(full_ds, train_classes, class_to_indices, CFG.n_way, CFG.k_shot, CFG.q_query, CFG.train_episodes), batch_size=1, shuffle=True)
    val_loader = DataLoader(FixedEpisodeDataset(val_ds, val_episodes), batch_size=1, shuffle=False)
    test_loader = DataLoader(FixedEpisodeDataset(val_ds, test_episodes), batch_size=1, shuffle=False)

    pd_full_ds = ImageFolderWithIndex(CFG.pd_root, transform=val_tfms)
    with open(pd_seen_json_path, "r") as f:
        all_pd_seen_episodes = json.load(f)
    with open(pd_unseen_json_path, "r") as f:
        all_pd_unseen_episodes = json.load(f)

    pd_seen_episodes = all_pd_seen_episodes[:CFG.eval_pd_episodes]
    pd_unseen_episodes = all_pd_unseen_episodes[:CFG.eval_pd_episodes]

    print(f"✓ PlantDoc images detected: {len(pd_full_ds)}", flush=True)
    print(f"✓ Episode counts - train: {CFG.train_episodes}, val: {len(val_episodes)}, "
          f"PV test: {len(test_episodes)}, PD SEEN: {len(pd_seen_episodes)}, "
          f"PD UNSEEN: {len(pd_unseen_episodes)}", flush=True)
    pd_seen_loader = DataLoader(FixedCrossDomainDataset(val_ds, pd_full_ds, pd_seen_episodes), batch_size=1, shuffle=False)
    pd_unseen_loader = DataLoader(FixedCrossDomainDataset(val_ds, pd_full_ds, pd_unseen_episodes), batch_size=1, shuffle=False)

    valid_variants = {
        "full", "no_dual_attn", "no_multiscale", "no_learnable_temp",
        "no_cutmix_mixup", "no_label_smoothing", "no_ortho_reg",
        "no_contrastive_loss", "reg_suite"
    }
    unknown_variants = set(VARIANTS_TO_RUN) - valid_variants
    if unknown_variants:
        raise ValueError(f"Unknown ablation variants: {sorted(unknown_variants)}")

    results = dict(COMPLETED_RESULTS)
    run_mode = "dry" if QUICK_DRY_RUN else "full"
    results_path = os.path.join(CFG.results_dir, f"ablation_results_{run_mode}_seed_{CFG.split_seed}.json")

    if os.path.exists(results_path):
        with open(results_path, "r") as f:
            saved_results = json.load(f)
        results.update(saved_results)
        print(f"✓ Resumed {len(saved_results)} saved result(s) from: {results_path}", flush=True)

    variants = [variant for variant in VARIANTS_TO_RUN if variant not in results]
    skipped = [variant for variant in VARIANTS_TO_RUN if variant in results]
    if skipped:
        print(f"✓ Skipping completed variants: {', '.join(skipped)}", flush=True)
    print(f"✓ Variants scheduled in this batch: {', '.join(variants) or 'none'}", flush=True)

    for var in variants:
        set_seed(CFG.split_seed)
        print(f"\n🚀 Running Ablation Variant: [{var}]...", flush=True)
        model = AblatedEfficientNetB0FSL(ablation_variant=var, pretrained=True).to(CFG.device)
        model = train_ablation_fsl(model, train_loader, val_loader, CFG, var)

        print("   Evaluating PlantVillage Test Set...", flush=True)
        pv_metrics = evaluate_ablation_model(model, test_loader, CFG)

        print("   Evaluating PlantDoc Cross-Domain (SEEN & UNSEEN)...", flush=True)
        pd_seen_acc = evaluate_plantdoc_episodes(model, pd_seen_loader, CFG, "PD SEEN")
        pd_unseen_acc = evaluate_plantdoc_episodes(model, pd_unseen_loader, CFG, "PD UNSEEN")

        results[var] = {"pv_acc": pv_metrics["pv_acc"], "pd_seen": pd_seen_acc, "pd_unseen": pd_unseen_acc}
        with open(results_path, "w") as f:
            json.dump(results, f, indent=2)
        print(f"   ✓ [{var}] -> PV: {pv_metrics['pv_acc']:.2f}% | PD SEEN: {pd_seen_acc:.2f}% | PD UNSEEN: {pd_unseen_acc:.2f}%", flush=True)
        print(f"   ✓ Partial results saved to: {results_path}", flush=True)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    generate_streamlined_ablation_table(results)

run_suite()


  🚀 FULL THESIS PRODUCTION RUN (QUICK_DRY_RUN = False)

✓ SEEN json:   pd_cross_SEEN_pd_mapping_v1_pairs56e21b1e78_rng200042_split42_n5_k5_q5_ep300.json
✓ UNSEEN json: pd_cross_UNSEEN_pd_mapping_v1_pairsb8e63f6b17_rng400042_split42_n5_k5_q5_ep300.json
✓ PlantVillage classes detected: 38
✓ Loaded 300/300 fixed val episodes from: pv_val_fixed_episodes_split42_n5_k5_q15_ep300.json
✓ Loaded 600/600 fixed test episodes from: pv_test_fixed_episodes_split42_n5_k5_q15_ep600.json
✓ PlantDoc images detected: 2920
✓ Episode counts - train: 600, val: 300, PV test: 600, PD SEEN: 300, PD UNSEEN: 300
✓ Variants scheduled in this batch: reg_suite

🚀 Running Ablation Variant: [reg_suite]...
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 112MB/s] 


   Epoch 1/40 - Val Acc: 87.37% (⭐ New Best Saved)
   Epoch 2/40 - Val Acc: 72.09%
   Epoch 3/40 - Val Acc: 76.04%
   Epoch 4/40 - Val Acc: 86.84%
   Epoch 5/40 - Val Acc: 88.90% (⭐ New Best Saved)
   Epoch 6/40 - Val Acc: 87.00%
   Epoch 7/40 - Val Acc: 89.14% (⭐ New Best Saved)
   Epoch 8/40 - Val Acc: 90.99% (⭐ New Best Saved)
   Epoch 9/40 - Val Acc: 91.70% (⭐ New Best Saved)
   Epoch 10/40 - Val Acc: 93.93% (⭐ New Best Saved)
   Epoch 11/40 - Val Acc: 92.52%
   Epoch 12/40 - Val Acc: 93.54%
   Epoch 13/40 - Val Acc: 92.41%
   Epoch 14/40 - Val Acc: 93.00%
   Epoch 15/40 - Val Acc: 91.76%
   Epoch 16/40 - Val Acc: 92.63%
   Epoch 17/40 - Val Acc: 93.15%
   Epoch 18/40 - Val Acc: 93.11%
   Early stopping triggered at epoch 18.
   Evaluating PlantVillage Test Set...
     [Evaluation] PV Acc: 90.96%
   Evaluating PlantDoc Cross-Domain (SEEN & UNSEEN)...
     [PD SEEN] 25/300 episodes | running acc: 50.08%
     [PD SEEN] 50/300 episodes | running acc: 46.80%
     [PD SEEN] 75/300 episo

# b